Implementation taken from https://github.com/pytorch/vision/blob/main/torchvision/models/swin_transformer.py

## Swin Transformer

In [1]:
import math
from functools import partial
from typing import Any, Callable, List, Optional

import torch
import torch.nn.functional as F
from torch import nn, Tensor

### Utils

In [2]:
class MLP(torch.nn.Sequential):
    """This block implements the multi-layer perceptron (MLP) module.

    Args:
        in_channels (int): Number of channels of the input
        hidden_channels (List[int]): List of the hidden channel dimensions
        norm_layer (Callable[..., torch.nn.Module], optional): Norm layer that will be stacked on top of the linear layer. If ``None`` this layer won't be used. Default: ``None``
        activation_layer (Callable[..., torch.nn.Module], optional): Activation function which will be stacked on top of the normalization layer (if not None), otherwise on top of the linear layer. If ``None`` this layer won't be used. Default: ``torch.nn.ReLU``
        inplace (bool, optional): Parameter for the activation layer, which can optionally do the operation in-place.
            Default is ``None``, which uses the respective default values of the ``activation_layer`` and Dropout layer.
        bias (bool): Whether to use bias in the linear layer. Default ``True``
        dropout (float): The probability for the dropout layer. Default: 0.0
    """

    def __init__(
        self,
        in_channels: int,
        hidden_channels: List[int],
        norm_layer: Optional[Callable[..., torch.nn.Module]] = None,
        activation_layer: Optional[Callable[..., torch.nn.Module]] = torch.nn.ReLU,
        inplace: Optional[bool] = None,
        bias: bool = True,
        dropout: float = 0.0,
    ):
        params = {} if inplace is None else {"inplace": inplace}

        layers = []
        in_dim = in_channels
        for hidden_dim in hidden_channels[:-1]:
            layers.append(torch.nn.Linear(in_dim, hidden_dim, bias=bias))
            if norm_layer is not None:
                layers.append(norm_layer(hidden_dim))
            layers.append(activation_layer(**params))
            layers.append(torch.nn.Dropout(dropout, **params))
            in_dim = hidden_dim

        layers.append(torch.nn.Linear(in_dim, hidden_channels[-1], bias=bias))
        layers.append(torch.nn.Dropout(dropout, **params))

        super().__init__(*layers)

In [3]:
class Permute(torch.nn.Module):
    """This module returns a view of the tensor input with its dimensions permuted.

    Args:
        dims (List[int]): The desired ordering of dimensions
    """

    def __init__(self, dims: List[int]):
        super().__init__()
        self.dims = dims

    def forward(self, x: Tensor) -> Tensor:
        return torch.permute(x, self.dims)

In [4]:
def stochastic_depth(
    input: Tensor, p: float, mode: str, training: bool = True
) -> Tensor:
    """
    Implements the Stochastic Depth from `"Deep Networks with Stochastic Depth"
    <https://arxiv.org/abs/1603.09382>`_ used for randomly dropping residual
    branches of residual architectures.

    Args:
        input (Tensor[N, ...]): The input tensor or arbitrary dimensions with the first one
                    being its batch i.e. a batch with ``N`` rows.
        p (float): probability of the input to be zeroed.
        mode (str): ``"batch"`` or ``"row"``.
                    ``"batch"`` randomly zeroes the entire input, ``"row"`` zeroes
                    randomly selected rows from the batch.
        training: apply stochastic depth if is ``True``. Default: ``True``

    Returns:
        Tensor[N, ...]: The randomly zeroed tensor.
    """
    if p < 0.0 or p > 1.0:
        raise ValueError(f"drop probability has to be between 0 and 1, but got {p}")
    if mode not in ["batch", "row"]:
        raise ValueError(f"mode has to be either 'batch' or 'row', but got {mode}")
    if not training or p == 0.0:
        return input

    survival_rate = 1.0 - p
    if mode == "row":
        size = [input.shape[0]] + [1] * (input.ndim - 1)
    else:
        size = [1] * input.ndim
    noise = torch.empty(size, dtype=input.dtype, device=input.device)
    noise = noise.bernoulli_(survival_rate)
    if survival_rate > 0.0:
        noise.div_(survival_rate)
    return input * noise


class StochasticDepth(nn.Module):
    """
    See :func:`stochastic_depth`.
    """

    def __init__(self, p: float, mode: str) -> None:
        super().__init__()
        self.p = p
        self.mode = mode

    def forward(self, input: Tensor) -> Tensor:
        return stochastic_depth(input, self.p, self.mode, self.training)

    def __repr__(self) -> str:
        s = f"{self.__class__.__name__}(p={self.p}, mode={self.mode})"
        return s

##### debug

In [ ]:
# Debug: Not part of code
import torch
from torch import nn
import torch.nn.functional as F

x = torch.ones((4, 3))  # batch of 4 samples, each with 3 features
print(x)
drop_layer = StochasticDepth(p=0.5, mode="row")
out = drop_layer(x)
print(out)

drop_layer = StochasticDepth(p=0.5, mode="batch")
out = drop_layer(x)
print(out)

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
tensor([[2., 2., 2.],
        [0., 0., 0.],
        [2., 2., 2.],
        [2., 2., 2.]])
tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]])


### Patch Merging Pad

In [6]:
def _patch_merging_pad(x: torch.Tensor, mask=False) -> torch.Tensor:
    """

    2 x 2 neighbours extracted for merge

    x : [1, 4, 4, 1]
    x.view(4,4) = tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])

    _patch_merging_pad(x) = tensor([[[[ 0,  4,  1,  5],
          [ 2,  6,  3,  7]],

         [[ 8, 12,  9, 13],
          [10, 14, 11, 15]]]]) : [1, 2, 2, 4]

    With uneven tensors, we must pad appropriately

    x: [1, 5, 5, 1]
    x.view(5,5) = tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19],
        [20, 21, 22, 23, 24]])

    _patch_merging_pad(x) = tensor([[[[ 4,  9,  0,  5],
          [ 1,  6,  2,  7],
          [ 3,  8,  4,  9]],

         [[14, 19, 10, 15],
          [11, 16, 12, 17],
          [13, 18, 14, 19]],

         [[24,  0, 20,  0],
          [21,  0, 22,  0],
          [23,  0, 24,  0]]]]): [1, 3, 3, 4]

    """
    if not mask:
        H, W, _ = x.shape[-3:]
        # Circular padding for x-axis (width)
        x = F.pad(x, (0, 0, W % 2, 0), mode="circular")

        # Zero padding for y-axis (height)
        x = F.pad(x, (0, 0, 0, 0, 0, H % 2))
        x0 = x[..., 0::2, 0::2, :]  # ... H/2 W/2 C
        x1 = x[..., 1::2, 0::2, :]  # ... H/2 W/2 C
        x2 = x[..., 0::2, 1::2, :]  # ... H/2 W/2 C
        x3 = x[..., 1::2, 1::2, :]  # ... H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # ... H/2 W/2 4*C
    else:
        H, W = x.shape[-2:]

        # Circular padding for x-axis (width)
        x = F.pad(x.float(), (0, W % 2), mode="circular").bool()

        # Zero padding for y-axis (height)
        x = F.pad(x.float(), (0, 0, 0, H % 2), value=0).bool()

        # Patch merging
        m0 = x[..., 0::2, 0::2]  # Top-left
        m1 = x[..., 1::2, 0::2]  # Bottom-left
        m2 = x[..., 0::2, 1::2]  # Top-right
        m3 = x[..., 1::2, 1::2]  # Bottom-right
        x = m0 & m1 & m2 & m3  # Logical OR for mask merging

    return x

##### debug

In [7]:
# Debug: Not part of code
N = 5
x = torch.arange(N * N).reshape(1, N, N, 1)
print(x.view(N, N))
x = _patch_merging_pad(x)
print(x)

# Are there 0 indices, because the filled value can possibly use the value at 0 index

# Mask
N = 5
mask = torch.ones(N * N).reshape(1, N, N).bool()
print(mask.view(N, N))
mask = _patch_merging_pad(mask, mask=True)
mask

tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19],
        [20, 21, 22, 23, 24]])
tensor([[[[ 4,  9,  0,  5],
          [ 1,  6,  2,  7],
          [ 3,  8,  4,  9]],

         [[14, 19, 10, 15],
          [11, 16, 12, 17],
          [13, 18, 14, 19]],

         [[24,  0, 20,  0],
          [21,  0, 22,  0],
          [23,  0, 24,  0]]]])
tensor([[True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True]])


tensor([[[ True,  True,  True],
         [ True,  True,  True],
         [False, False, False]]])

### Patch Merging

In [8]:
class PatchMerging(nn.Module):
    """Patch Merging Layer.
    Args:
        dim (int): Number of input channels.
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
    """

    def __init__(self, dim: int, norm_layer: Callable[..., nn.Module] = nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x: Tensor):
        """
        Args:
            x (Tensor): input tensor with expected layout of [..., H, W, C]
        Returns:
            Tensor with layout of [..., H/2, W/2, 2*C]
        """
        x = _patch_merging_pad(x)
        x = self.norm(x)
        x = self.reduction(x)  # ... H/2 W/2 2*C
        return x


class PatchMergingV2(nn.Module):
    """Patch Merging Layer for Swin Transformer V2.
    Args:
        dim (int): Number of input channels.
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
    """

    def __init__(self, dim: int, norm_layer: Callable[..., nn.Module] = nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(2 * dim)  # difference

    def forward(self, x: Tensor):
        """
        Args:
            x (Tensor): input tensor with expected layout of [..., H, W, C]
        Returns:
            Tensor with layout of [..., H/2, W/2, 2*C]
        """
        x = _patch_merging_pad(x)
        x = self.reduction(x)  # ... H/2 W/2 2*C
        x = self.norm(x)
        return x

##### debug

In [9]:
# debug
pm = PatchMerging(dim=1)
N = 5
x = torch.arange(N * N).reshape(1, N, N, 1)
print(x)
print(x.shape)
res = pm(x.float())
print(res)
print(res.shape)

tensor([[[[ 0],
          [ 1],
          [ 2],
          [ 3],
          [ 4]],

         [[ 5],
          [ 6],
          [ 7],
          [ 8],
          [ 9]],

         [[10],
          [11],
          [12],
          [13],
          [14]],

         [[15],
          [16],
          [17],
          [18],
          [19]],

         [[20],
          [21],
          [22],
          [23],
          [24]]]])
torch.Size([1, 5, 5, 1])
tensor([[[[ 0.1722,  0.7117],
          [-0.7021,  1.0294],
          [-0.7021,  1.0294]],

         [[ 0.1722,  0.7117],
          [-0.7021,  1.0294],
          [-0.7021,  1.0294]],

         [[ 0.6154, -1.0395],
          [ 0.5049, -1.0149],
          [ 0.5070, -1.0156]]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([1, 3, 3, 2])


In [10]:
# debug
pm = PatchMergingV2(dim=1)
N = 5
x = torch.arange(N * N).reshape(1, N, N, 1)
pm(x.float())

tensor([[[[-1.0000,  1.0000],
          [-1.0000,  1.0000],
          [-1.0000,  1.0000]],

         [[-1.0000,  1.0000],
          [-1.0000,  1.0000],
          [-1.0000,  1.0000]],

         [[-1.0000,  1.0000],
          [-1.0000,  1.0000],
          [-1.0000,  1.0000]]]], grad_fn=<NativeLayerNormBackward0>)

### Shifted Window Attention - Core

In [30]:
def shifted_window_attention(
    input: Tensor,
    qkv_weight: Tensor,
    proj_weight: Tensor,
    relative_position_bias: Tensor,
    window_size: List[int],
    num_heads: int,
    shift_size: List[int],
    attention_dropout: float = 0.0,
    dropout: float = 0.0,
    qkv_bias: Optional[Tensor] = None,
    proj_bias: Optional[Tensor] = None,
    logit_scale: Optional[torch.Tensor] = None,
    training: bool = True,
) -> Tensor:
    """
    Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.
    Args:
        input (Tensor[N, H, W, C]): The input tensor or 4-dimensions.
        qkv_weight (Tensor[in_dim, out_dim]): The weight tensor of query, key, value.
        proj_weight (Tensor[out_dim, out_dim]): The weight tensor of projection.
        relative_position_bias (Tensor): The learned relative position bias added to attention.
        window_size (List[int]): Window size.
        num_heads (int): Number of attention heads.
        shift_size (List[int]): Shift size for shifted window attention.
        attention_dropout (float): Dropout ratio of attention weight. Default: 0.0.
        dropout (float): Dropout ratio of output. Default: 0.0.
        qkv_bias (Tensor[out_dim], optional): The bias tensor of query, key, value. Default: None.
        proj_bias (Tensor[out_dim], optional): The bias tensor of projection. Default: None.
        logit_scale (Tensor[out_dim], optional): Logit scale of cosine attention for Swin Transformer V2. Default: None.
        training (bool, optional): Training flag used by the dropout parameters. Default: True.
    Returns:
        Tensor[N, H, W, C]: The output tensor after shifted window attention.
    """
    B, H, W, C = input.shape
    # pad feature maps to multiples of window size
    pad_r = (window_size[1] - W % window_size[1]) % window_size[1]
    pad_b = (window_size[0] - H % window_size[0]) % window_size[0]

    x = F.pad(input, (0, 0, pad_r, 0), mode="circular")
    x = F.pad(x, (0, 0, 0, 0, 0, pad_b))

    _, pad_H, pad_W, _ = x.shape

    shift_size = shift_size.copy()
    # If window size is larger than feature size, there is no need to shift window
    if window_size[0] >= pad_H:
        shift_size[0] = 0
    if window_size[1] >= pad_W:
        shift_size[1] = 0

    # cyclic shift
    if sum(shift_size) > 0:
        x = torch.roll(x, shifts=(-shift_size[0], -shift_size[1]), dims=(1, 2))

    # partition windows
    num_windows = (pad_H // window_size[0]) * (pad_W // window_size[1])
    x = x.view(
        B,
        pad_H // window_size[0],
        window_size[0],
        pad_W // window_size[1],
        window_size[1],
        C,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).reshape(
        B * num_windows, window_size[0] * window_size[1], C
    )  # B*nW, Ws*Ws, C

    # multi-head attention
    if logit_scale is not None and qkv_bias is not None:
        qkv_bias = qkv_bias.clone()
        length = qkv_bias.numel() // 3
        # zero out the bias for the Key since it would break cosine similarity computation
        qkv_bias[length : 2 * length].zero_()
    qkv = F.linear(x, qkv_weight, qkv_bias)  # Q, K, V
    qkv = qkv.reshape(x.size(0), x.size(1), 3, num_heads, C // num_heads).permute(
        2, 0, 3, 1, 4
    )  # 3, B*nW, num_heads, Ws*Ws, C//num_heads
    q, k, v = qkv[0], qkv[1], qkv[2]
    if logit_scale is not None:
        # cosine attention is scale-invariant and thus we use logit_scale to scale the attention
        attn = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1)
        logit_scale = torch.clamp(logit_scale, max=math.log(100.0)).exp()
        attn = attn * logit_scale
    else:
        q = q * (C // num_heads) ** -0.5
        attn = q.matmul(k.transpose(-2, -1))

    # add relative position bias
    attn = attn + relative_position_bias  # B*nW, num_heads, Ws*Ws, Ws*Ws

    if sum(shift_size) > 0:
        # generate attention mask
        attn_mask = x.new_zeros((pad_H, pad_W))
        h_slices = (
            (0, -window_size[0]),
            (-window_size[0], -shift_size[0]),
            (-shift_size[0], None),
        )
        w_slices = (
            (0, -window_size[1]),
            (-window_size[1], -shift_size[1]),
            (-shift_size[1], None),
        )
        count = 0
        for h in h_slices:
            for w in w_slices:
                attn_mask[h[0] : h[1], w[0] : w[1]] = count
                count += 1
        attn_mask = attn_mask.view(
            pad_H // window_size[0],
            window_size[0],
            pad_W // window_size[1],
            window_size[1],
        )
        attn_mask = attn_mask.permute(0, 2, 1, 3).reshape(
            num_windows, window_size[0] * window_size[1]
        )
        attn_mask = attn_mask.unsqueeze(1) - attn_mask.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(
            attn_mask == 0, float(0.0)
        )
        attn = attn.view(
            x.size(0) // num_windows, num_windows, num_heads, x.size(1), x.size(1)
        )
        attn = attn + attn_mask.unsqueeze(1).unsqueeze(0)
        attn = attn.view(-1, num_heads, x.size(1), x.size(1))

    attn = F.softmax(attn, dim=-1)
    attn = F.dropout(attn, p=attention_dropout, training=training)

    x = attn.matmul(v).transpose(1, 2).reshape(x.size(0), x.size(1), C)
    x = F.linear(x, proj_weight, proj_bias)
    x = F.dropout(x, p=dropout, training=training)

    # reverse windows
    x = x.view(
        B,
        pad_H // window_size[0],
        pad_W // window_size[1],
        window_size[0],
        window_size[1],
        C,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).reshape(B, pad_H, pad_W, C)

    # reverse cyclic shift
    if sum(shift_size) > 0:
        x = torch.roll(x, shifts=(shift_size[0], shift_size[1]), dims=(1, 2))

    # unpad features
    x = x[:, :H, :W, :].contiguous()
    return x

##### debug

In [31]:
# debug: Padding
window_size = [7, 7]
W = 100
# Need to find how much more to pad to make it divisible by window size
pad_r = (window_size[1] - W % window_size[1]) % window_size[1]
print(pad_r)

# if divisible by window size, then pad_r = 0, so we need outer modulo
window_size = [5, 5]
W = 100
pad_r = window_size[1] - W % window_size[1]  # if we didnt have it
pad_r  # This should be 0 but gives 5

5


5

In [32]:
# debug: roll
"""
a, b, c < D in terms of shape

a b    ->   D c
c D         b a
"""
shift_size = [2, 2]
N = 4
x = torch.arange(N * N).reshape(1, N, N, 1)
print(x.view(N, N))
x = torch.roll(
    x, shifts=(-shift_size[0], -shift_size[1]), dims=(1, 2)
)  # Shifting across H, W
print(x.view(N, N))

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])
tensor([[10, 11,  8,  9],
        [14, 15, 12, 13],
        [ 2,  3,  0,  1],
        [ 6,  7,  4,  5]])


In [33]:
# debug: num_windows and reshape
window_size = [2, 2]
pad_H = 6
pad_W = 6
B = 1
C = 1
x = torch.arange(pad_H * pad_W).reshape(1, pad_H, pad_W, 1)
num_windows = (pad_H // window_size[0]) * (pad_W // window_size[1])
print(num_windows)
print(x.view(pad_H, pad_W))
print(x.shape)
x = x.view(
    B,
    pad_H // window_size[0],
    window_size[0],
    pad_W // window_size[1],
    window_size[1],
    C,
)
print(x.shape)
x = x.permute(0, 1, 3, 2, 4, 5).reshape(
    B * num_windows, window_size[0] * window_size[1], C
)  # B*nW, Ws*Ws, C
print(x.shape)  # Number of windows x pixels in window x channels for each pixel

9
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17],
        [18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29],
        [30, 31, 32, 33, 34, 35]])
torch.Size([1, 6, 6, 1])
torch.Size([1, 3, 2, 3, 2, 1])
torch.Size([9, 4, 1])


In [34]:
# debug: multi-head attention
logit_scale = torch.tensor(1.0)
qkv_bias = None
num_heads = 3
C = 18
num_windows = 4
B = 1
window_size = [2, 2]
qkv = nn.Linear(C, C * 3)
qkv_weight = qkv.weight
qkv_bias = qkv.bias
x = torch.randn(B * num_windows, window_size[0] * window_size[1], C)
print(x.shape)
if logit_scale is not None and qkv_bias is not None:
    qkv_bias = qkv_bias.clone()
    length = qkv_bias.numel() // 3
    qkv_bias[length : 2 * length].zero_()
    print(qkv_bias)
qkv = F.linear(x, qkv_weight, qkv_bias)  # Q, K, V
print(qkv.shape)
print(qkv.reshape(x.size(0), x.size(1), 3, num_heads, C // num_heads).shape)
qkv = qkv.reshape(x.size(0), x.size(1), 3, num_heads, C // num_heads).permute(
    2, 0, 3, 1, 4
)  # 3, B*nW, num_heads, Ws*Ws, C//num_heads
print(qkv.shape)
q, k, v = qkv[0], qkv[1], qkv[2]
if logit_scale is not None:
    # cosine attention
    attn = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1)
    logit_scale = torch.clamp(logit_scale, max=math.log(100.0)).exp()
    attn = attn * logit_scale
else:
    q = q * (C // num_heads) ** -0.5
    attn = q.matmul(k.transpose(-2, -1))
print(attn.shape)  # B*nW, num_heads, Ws*Ws, Ws*Ws

torch.Size([4, 4, 18])
tensor([ 0.0003,  0.1443,  0.2030, -0.1716, -0.2337,  0.0840,  0.1388, -0.0894,
        -0.2319, -0.1783, -0.1503, -0.0923, -0.0746,  0.1159,  0.0671,  0.1345,
        -0.0727,  0.1473,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000, -0.0218, -0.0490, -0.1866,  0.1042,
        -0.0180, -0.0438, -0.1432,  0.0623,  0.1746,  0.0706,  0.0582,  0.0728,
         0.1130, -0.1290,  0.2156,  0.1830, -0.2000, -0.0369],
       grad_fn=<CopySlices>)
torch.Size([4, 4, 54])
torch.Size([4, 4, 3, 3, 6])
torch.Size([3, 4, 3, 4, 6])
torch.Size([4, 3, 4, 4])


In [35]:
# debug: softmax and dropout
attention_dropout = 0.0
training = True
dropout = 0.0
proj = nn.Linear(C, C)
proj_weight = proj.weight
proj_bias = proj.bias
pad_H = 4
pad_W = 4
attn = F.softmax(attn, dim=-1)
attn = F.dropout(attn, p=attention_dropout, training=training)

print(attn.shape)
print(v.shape)
x = attn.matmul(v).transpose(1, 2).reshape(x.size(0), x.size(1), C)
print(x.shape)
x = F.linear(x, proj_weight, proj_bias)
x = F.dropout(x, p=dropout, training=training)
print(x.shape)

# reverse windows
x = x.view(
    B,
    pad_H // window_size[0],
    pad_W // window_size[1],
    window_size[0],
    window_size[1],
    C,
)
x = x.permute(0, 1, 3, 2, 4, 5).reshape(B, pad_H, pad_W, C)
print(x.shape)
# reverse cyclic shift
if sum(shift_size) > 0:
    x = torch.roll(x, shifts=(shift_size[0], shift_size[1]), dims=(1, 2))

torch.Size([4, 3, 4, 4])
torch.Size([4, 3, 4, 6])
torch.Size([4, 4, 18])
torch.Size([4, 4, 18])
torch.Size([1, 4, 4, 18])


In [36]:
# debug: attention mask for shifted window attention
window_size = [3, 3]
shift_size = [2, 2]
pad_H = 6
pad_W = 6
B = 1
num_windows = pad_H // window_size[0] * pad_W // window_size[1]
x = torch.randn(B * num_windows, window_size[0] * window_size[1], C)
attn = torch.randn(
    B * num_windows,
    num_heads,
    window_size[0] * window_size[1],
    window_size[0] * window_size[1],
)
print(attn.shape)
attn_mask = x.new_zeros((pad_H, pad_W))
print(attn_mask)
h_slices = (
    (0, -window_size[0]),
    (-window_size[0], -shift_size[0]),
    (-shift_size[0], None),
)
w_slices = (
    (0, -window_size[1]),
    (-window_size[1], -shift_size[1]),
    (-shift_size[1], None),
)
print(h_slices)
print(w_slices)
count = 0
for h in h_slices:
    for w in w_slices:
        attn_mask[h[0] : h[1], w[0] : w[1]] = count
        count += 1
print(attn_mask)
attn_mask = attn_mask.view(
    pad_H // window_size[0],
    window_size[0],
    pad_W // window_size[1],
    window_size[1],
)
attn_mask = attn_mask.permute(0, 2, 1, 3).reshape(
    num_windows, window_size[0] * window_size[1]
)
print(attn_mask.shape)
print(attn_mask)
attn_mask = attn_mask.unsqueeze(1) - attn_mask.unsqueeze(2)
print(attn_mask.shape)
print(attn_mask)
attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(
    attn_mask == 0, float(0.0)
)
print(attn_mask)
attn = attn.view(x.size(0) // num_windows, num_windows, num_heads, x.size(1), x.size(1))
print(attn.shape)
attn = attn + attn_mask.unsqueeze(1).unsqueeze(0)
attn = attn.view(-1, num_heads, x.size(1), x.size(1))

torch.Size([4, 3, 9, 9])
tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
((0, -3), (-3, -2), (-2, None))
((0, -3), (-3, -2), (-2, None))
tensor([[0., 0., 0., 1., 2., 2.],
        [0., 0., 0., 1., 2., 2.],
        [0., 0., 0., 1., 2., 2.],
        [3., 3., 3., 4., 5., 5.],
        [6., 6., 6., 7., 8., 8.],
        [6., 6., 6., 7., 8., 8.]])
torch.Size([4, 9])
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 2., 2., 1., 2., 2., 1., 2., 2.],
        [3., 3., 3., 6., 6., 6., 6., 6., 6.],
        [4., 5., 5., 7., 8., 8., 7., 8., 8.]])
torch.Size([4, 9, 9])
tensor([[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  

In [37]:
# debug: Put it all together
window_size = [3, 3]
H, W = 5, 5
C = 18
B = 1
x = torch.randn(B, H, W, C)
print(x.shape)
qkv = nn.Linear(C, C * 3)
proj = nn.Linear(C, C)
qkv_weight = qkv.weight
qkv_bias = qkv.bias
proj_weight = proj.weight
proj_bias = proj.bias
num_heads = 3
attention_dropout = 0.0
dropout = 0.0
logit_scale = torch.tensor(1.0)
training = True
relative_position_bias = torch.randn(
    window_size[0] * window_size[1], window_size[0] * window_size[1]
)

# Case 1: No shift
shift_size = [0, 0]
res = shifted_window_attention(
    x,
    qkv_weight,
    proj_weight,
    relative_position_bias,
    window_size,
    num_heads,
    shift_size,
    attention_dropout,
    dropout,
    qkv_bias,
    proj_bias,
    logit_scale,
    training,
)
print(res.shape)


# Case 2: Shift
shift_size = [2, 2]
res = shifted_window_attention(
    x,
    qkv_weight,
    proj_weight,
    relative_position_bias,
    window_size,
    num_heads,
    shift_size,
    attention_dropout,
    dropout,
    qkv_bias,
    proj_bias,
    logit_scale,
    training,
)
print(res.shape)

torch.Size([1, 5, 5, 18])
torch.Size([1, 5, 5, 18])
torch.Size([1, 5, 5, 18])


### Shifted Window Attention

In [38]:
# untested
def _get_relative_position_bias(
    relative_position_bias_table: torch.Tensor,
    relative_position_index: torch.Tensor,
    window_size: List[int],
) -> torch.Tensor:
    N = window_size[0] * window_size[1]
    relative_position_bias = relative_position_bias_table[relative_position_index]  # type: ignore[index]
    relative_position_bias = relative_position_bias.view(N, N, -1)
    relative_position_bias = (
        relative_position_bias.permute(2, 0, 1).contiguous().unsqueeze(0)
    )
    return relative_position_bias

In [ ]:
class ShiftedWindowAttention(nn.Module):
    """
    See :func:`shifted_window_attention`.
    """

    def __init__(
        self,
        dim: int,
        window_size: List[int],
        shift_size: List[int],
        num_heads: int,
        qkv_bias: bool = True,
        proj_bias: bool = True,
        attention_dropout: float = 0.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        if len(window_size) != 2 or len(shift_size) != 2:
            raise ValueError("window_size and shift_size must be of length 2")
        self.window_size = window_size
        self.shift_size = shift_size
        self.num_heads = num_heads
        self.attention_dropout = attention_dropout
        self.dropout = dropout

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim, bias=proj_bias)

        self.define_relative_position_bias_table()
        self.define_relative_position_index()

    def define_relative_position_bias_table(self):
        # define a parameter table of relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros(
                (2 * self.window_size[0] - 1) * (2 * self.window_size[1] - 1),
                self.num_heads,
            )
        )  # 2*Wh-1 * 2*Ww-1, nH
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def define_relative_position_index(self):
        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(
            torch.meshgrid(coords_h, coords_w, indexing="ij")
        )  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = (
            coords_flatten[:, :, None] - coords_flatten[:, None, :]
        )  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(
            1, 2, 0
        ).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1).flatten()  # Wh*Ww*Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index)

    def get_relative_position_bias(self) -> torch.Tensor:
        return _get_relative_position_bias(
            self.relative_position_bias_table,
            self.relative_position_index,
            self.window_size,  # type: ignore[arg-type]
        )

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x (Tensor): Tensor with layout of [B, H, W, C]
        Returns:
            Tensor with same layout as input, i.e. [B, H, W, C]
        """
        relative_position_bias = self.get_relative_position_bias()
        return shifted_window_attention(
            x,
            self.qkv.weight,
            self.proj.weight,
            relative_position_bias,
            self.window_size,
            self.num_heads,
            shift_size=self.shift_size,
            attention_dropout=self.attention_dropout,
            dropout=self.dropout,
            qkv_bias=self.qkv.bias,
            proj_bias=self.proj.bias,
            training=self.training,
        )


class ShiftedWindowAttentionV2(ShiftedWindowAttention):
    """
    See :func:`shifted_window_attention_v2`.
    """

    def __init__(
        self,
        dim: int,
        window_size: List[int],
        shift_size: List[int],
        num_heads: int,
        qkv_bias: bool = True,
        proj_bias: bool = True,
        attention_dropout: float = 0.0,
        dropout: float = 0.0,
    ):
        super().__init__(
            dim,
            window_size,
            shift_size,
            num_heads,
            qkv_bias=qkv_bias,
            proj_bias=proj_bias,
            attention_dropout=attention_dropout,
            dropout=dropout,
        )

        self.logit_scale = nn.Parameter(torch.log(10 * torch.ones((num_heads, 1, 1))))
        # mlp to generate continuous relative position bias
        self.cpb_mlp = nn.Sequential(
            nn.Linear(2, 512, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_heads, bias=False),
        )
        if qkv_bias:
            length = self.qkv.bias.numel() // 3
            self.qkv.bias[length : 2 * length].data.zero_()

    def define_relative_position_bias_table(self):
        # get relative_coords_table
        relative_coords_h = torch.arange(
            -(self.window_size[0] - 1), self.window_size[0], dtype=torch.float32
        )
        relative_coords_w = torch.arange(
            -(self.window_size[1] - 1), self.window_size[1], dtype=torch.float32
        )
        relative_coords_table = torch.stack(
            torch.meshgrid([relative_coords_h, relative_coords_w], indexing="ij")
        )
        relative_coords_table = (
            relative_coords_table.permute(1, 2, 0).contiguous().unsqueeze(0)
        )  # 1, 2*Wh-1, 2*Ww-1, 2

        relative_coords_table[:, :, :, 0] /= self.window_size[0] - 1
        relative_coords_table[:, :, :, 1] /= self.window_size[1] - 1

        relative_coords_table *= 8  # normalize to -8, 8
        relative_coords_table = (
            torch.sign(relative_coords_table)
            * torch.log2(torch.abs(relative_coords_table) + 1.0)
            / 3.0
        )
        self.register_buffer("relative_coords_table", relative_coords_table)

    def get_relative_position_bias(self) -> torch.Tensor:
        relative_position_bias = _get_relative_position_bias(
            self.cpb_mlp(self.relative_coords_table).view(-1, self.num_heads),
            self.relative_position_index,  # type: ignore[arg-type]
            self.window_size,
        )
        relative_position_bias = 16 * torch.sigmoid(relative_position_bias)
        return relative_position_bias

    def forward(self, x: Tensor):
        """
        Args:
            x (Tensor): Tensor with layout of [B, H, W, C]
        Returns:
            Tensor with same layout as input, i.e. [B, H, W, C]
        """
        relative_position_bias = self.get_relative_position_bias()
        return shifted_window_attention(
            x,
            self.qkv.weight,
            self.proj.weight,
            relative_position_bias,
            self.window_size,
            self.num_heads,
            shift_size=self.shift_size,
            attention_dropout=self.attention_dropout,
            dropout=self.dropout,
            qkv_bias=self.qkv.bias,
            proj_bias=self.proj.bias,
            logit_scale=self.logit_scale,
            training=self.training,
        )

##### Debug


In [70]:
# V1
# Define bias table
window_size = [3, 3]
num_heads = 3
relative_position_bias_table = nn.Parameter(
    torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads)
)  # 2*Wh-1 * 2*Ww-1, nH

nn.init.trunc_normal_(relative_position_bias_table, std=0.02)
print(relative_position_bias_table.shape)
relative_position_bias_table

torch.Size([25, 3])


Parameter containing:
tensor([[-0.0487, -0.0400, -0.0060],
        [ 0.0205,  0.0078,  0.0203],
        [ 0.0072,  0.0298, -0.0361],
        [-0.0445, -0.0085, -0.0089],
        [-0.0121,  0.0073,  0.0227],
        [ 0.0060,  0.0211, -0.0242],
        [-0.0011, -0.0046, -0.0031],
        [-0.0055, -0.0008,  0.0113],
        [ 0.0162, -0.0130,  0.0194],
        [-0.0175, -0.0290, -0.0241],
        [ 0.0171,  0.0328, -0.0008],
        [-0.0255, -0.0137, -0.0390],
        [ 0.0065, -0.0274,  0.0209],
        [ 0.0042, -0.0267, -0.0283],
        [-0.0042, -0.0180, -0.0374],
        [-0.0208, -0.0174,  0.0069],
        [-0.0227, -0.0064, -0.0143],
        [ 0.0186,  0.0135,  0.0150],
        [ 0.0021, -0.0149,  0.0012],
        [-0.0015, -0.0152, -0.0197],
        [-0.0022, -0.0130, -0.0125],
        [ 0.0032, -0.0303,  0.0196],
        [ 0.0102,  0.0162,  0.0315],
        [ 0.0460, -0.0088,  0.0010],
        [ 0.0203, -0.0215, -0.0141]], requires_grad=True)

In [69]:
# V1
# Define relative position index
coords_h = torch.arange(window_size[0])
coords_w = torch.arange(window_size[1])
coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing="ij"))  # 2, Wh, Ww
print(coords)
coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
print(coords_flatten)
relative_coords = (
    coords_flatten[:, :, None] - coords_flatten[:, None, :]
)  # 2, Wh*Ww, Wh*Ww
print(relative_coords)  # Range: -(Ww-1) : Ww-1 ; -(Wh-1) : Wh-1
relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
print(relative_coords)
relative_coords[:, :, 0] += window_size[0] - 1  # shift to start from 0
print(relative_coords)  # Range: 0 : 2*(Ww-1) ; -(Wh-1) : Wh-1
relative_coords[:, :, 1] += window_size[1] - 1
print(relative_coords)  # Range: 0 : 2*(Ww-1) ; 0 : 2*(Wh-1)
relative_coords[:, :, 0] *= 2 * window_size[1] - 1
print(relative_coords)  # Range: 0 : 2*(Ww-1) * (2*Wh-1) ; 0 : 2*(Wh-1)
relative_position_index = relative_coords.sum(-1).flatten()  # Wh*Ww*Wh*Ww
print(
    relative_position_index
)  # Range: 0 : 2*(Ww-1) * (2*Wh-1) + 2*(Wh-1) = 0: (2Ww-1)(2Wh-1) - 1
# 2*(2WwWh-Ww-2Wh+1) + 2Wh -2 = 4WwWh - 2Ww - 4Wh + 2 + 2Wh - 2 = 4WwWh - 2Ww - 2Wh = (2Ww-1)(2Wh-1) - 1
print(relative_position_index.shape)
print(relative_position_index.max())
print(relative_position_index.min())
print((2 * window_size[0] - 1) * (2 * window_size[1] - 1) - 1)

tensor([[[0, 0, 0],
         [1, 1, 1],
         [2, 2, 2]],

        [[0, 1, 2],
         [0, 1, 2],
         [0, 1, 2]]])
tensor([[0, 0, 0, 1, 1, 1, 2, 2, 2],
        [0, 1, 2, 0, 1, 2, 0, 1, 2]])
tensor([[[ 0,  0,  0, -1, -1, -1, -2, -2, -2],
         [ 0,  0,  0, -1, -1, -1, -2, -2, -2],
         [ 0,  0,  0, -1, -1, -1, -2, -2, -2],
         [ 1,  1,  1,  0,  0,  0, -1, -1, -1],
         [ 1,  1,  1,  0,  0,  0, -1, -1, -1],
         [ 1,  1,  1,  0,  0,  0, -1, -1, -1],
         [ 2,  2,  2,  1,  1,  1,  0,  0,  0],
         [ 2,  2,  2,  1,  1,  1,  0,  0,  0],
         [ 2,  2,  2,  1,  1,  1,  0,  0,  0]],

        [[ 0, -1, -2,  0, -1, -2,  0, -1, -2],
         [ 1,  0, -1,  1,  0, -1,  1,  0, -1],
         [ 2,  1,  0,  2,  1,  0,  2,  1,  0],
         [ 0, -1, -2,  0, -1, -2,  0, -1, -2],
         [ 1,  0, -1,  1,  0, -1,  1,  0, -1],
         [ 2,  1,  0,  2,  1,  0,  2,  1,  0],
         [ 0, -1, -2,  0, -1, -2,  0, -1, -2],
         [ 1,  0, -1,  1,  0, -1,  1,  0, -1],


In [84]:
# V2
# Relative position bias table
relative_coords_h = torch.arange(
    -(window_size[0] - 1), window_size[0], dtype=torch.float32
)
print(relative_coords_h.shape)  # 2*Wh-1
print(relative_coords_h)
relative_coords_w = torch.arange(
    -(window_size[1] - 1), window_size[1], dtype=torch.float32
)
print(relative_coords_w)  # 2*Ww-1
relative_coords_table = torch.stack(
    torch.meshgrid([relative_coords_h, relative_coords_w], indexing="ij")
)
print(relative_coords_table)  # 2, 2*Wh-1, 2*Ww-1
relative_coords_table = (
    relative_coords_table.permute(1, 2, 0).contiguous().unsqueeze(0)
)  # 1, 2*Wh-1, 2*Ww-1, 2
print(relative_coords_table)

relative_coords_table[:, :, :, 0] /= window_size[0] - 1
relative_coords_table[:, :, :, 1] /= window_size[1] - 1
print(relative_coords_table)
relative_coords_table *= 8  # normalize to -8, 8
print(relative_coords_table)
relative_coords_table = (
    torch.sign(relative_coords_table)
    * torch.log2(torch.abs(relative_coords_table) + 1.0)
    / 3.0
)
print(relative_coords_table.shape)  # 1, 2*Wh-1, 2*Ww-1, 2
print(relative_coords_table)

torch.Size([5])
tensor([-2., -1.,  0.,  1.,  2.])
tensor([-2., -1.,  0.,  1.,  2.])
tensor([[[-2., -2., -2., -2., -2.],
         [-1., -1., -1., -1., -1.],
         [ 0.,  0.,  0.,  0.,  0.],
         [ 1.,  1.,  1.,  1.,  1.],
         [ 2.,  2.,  2.,  2.,  2.]],

        [[-2., -1.,  0.,  1.,  2.],
         [-2., -1.,  0.,  1.,  2.],
         [-2., -1.,  0.,  1.,  2.],
         [-2., -1.,  0.,  1.,  2.],
         [-2., -1.,  0.,  1.,  2.]]])
tensor([[[[-2., -2.],
          [-2., -1.],
          [-2.,  0.],
          [-2.,  1.],
          [-2.,  2.]],

         [[-1., -2.],
          [-1., -1.],
          [-1.,  0.],
          [-1.,  1.],
          [-1.,  2.]],

         [[ 0., -2.],
          [ 0., -1.],
          [ 0.,  0.],
          [ 0.,  1.],
          [ 0.,  2.]],

         [[ 1., -2.],
          [ 1., -1.],
          [ 1.,  0.],
          [ 1.,  1.],
          [ 1.,  2.]],

         [[ 2., -2.],
          [ 2., -1.],
          [ 2.,  0.],
          [ 2.,  1.],
          [ 2., 

In [88]:
cpb_mlp = nn.Sequential(
    nn.Linear(2, 512, bias=True),
    nn.ReLU(inplace=True),
    nn.Linear(512, num_heads, bias=False),
)

relative_position_bias = _get_relative_position_bias(
    cpb_mlp(relative_coords_table).view(-1, num_heads),
    relative_position_index,  # type: ignore[arg-type]
    window_size,
)
relative_position_bias = 16 * torch.sigmoid(relative_position_bias)
relative_position_bias

tensor([[[[7.7536, 8.1748, 8.2255, 8.1098, 8.5426, 8.6186, 8.2236, 8.4812,
           8.5898],
          [7.5764, 7.7536, 8.1748, 7.8748, 8.1098, 8.5426, 8.0779, 8.2236,
           8.4812],
          [7.6722, 7.5764, 7.7536, 7.9189, 7.8748, 8.1098, 8.0926, 8.0779,
           8.2236],
          [7.2362, 7.4706, 7.5658, 7.7536, 8.1748, 8.2255, 8.1098, 8.5426,
           8.6186],
          [7.2795, 7.2362, 7.4706, 7.5764, 7.7536, 8.1748, 7.8748, 8.1098,
           8.5426],
          [7.3987, 7.2795, 7.2362, 7.6722, 7.5764, 7.7536, 7.9189, 7.8748,
           8.1098],
          [7.0868, 7.1727, 7.2680, 7.2362, 7.4706, 7.5658, 7.7536, 8.1748,
           8.2255],
          [7.1719, 7.0868, 7.1727, 7.2795, 7.2362, 7.4706, 7.5764, 7.7536,
           8.1748],
          [7.3150, 7.1719, 7.0868, 7.3987, 7.2795, 7.2362, 7.6722, 7.5764,
           7.7536]],

         [[8.1871, 7.5551, 7.4606, 8.6038, 8.2089, 7.9939, 8.5970, 8.3938,
           8.2255],
          [8.4873, 8.1871, 7.5551, 8.3858, 8.603

### Swin Transformer

In [ ]:
class SwinTransformerBlock(nn.Module):
    """
    Swin Transformer Block.
    Args:
        dim (int): Number of input channels.
        num_heads (int): Number of attention heads.
        window_size (List[int]): Window size.
        shift_size (List[int]): Shift size for shifted window attention.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4.0.
        dropout (float): Dropout rate. Default: 0.0.
        attention_dropout (float): Attention dropout rate. Default: 0.0.
        stochastic_depth_prob: (float): Stochastic depth rate. Default: 0.0.
        norm_layer (nn.Module): Normalization layer.  Default: nn.LayerNorm.
        attn_layer (nn.Module): Attention layer. Default: ShiftedWindowAttention
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        window_size: List[int],
        shift_size: List[int],
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        attention_dropout: float = 0.0,
        stochastic_depth_prob: float = 0.0,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        attn_layer: Callable[..., nn.Module] = ShiftedWindowAttention,
    ):
        super().__init__()

        self.norm1 = norm_layer(dim)
        self.attn = attn_layer(
            dim,
            window_size,
            shift_size,
            num_heads,
            attention_dropout=attention_dropout,
            dropout=dropout,
        )
        self.stochastic_depth = StochasticDepth(stochastic_depth_prob, "row")
        self.norm2 = norm_layer(dim)
        self.mlp = MLP(
            dim,
            [int(dim * mlp_ratio), dim],
            activation_layer=nn.GELU,
            inplace=None,
            dropout=dropout,
        )

        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.normal_(m.bias, std=1e-6)

    def forward(self, x: Tensor):
        x = x + self.stochastic_depth(self.attn(self.norm1(x)))
        x = x + self.stochastic_depth(self.mlp(self.norm2(x)))
        return x


class SwinTransformerBlockV2(SwinTransformerBlock):
    """
    Swin Transformer V2 Block.
    Args:
        dim (int): Number of input channels.
        num_heads (int): Number of attention heads.
        window_size (List[int]): Window size.
        shift_size (List[int]): Shift size for shifted window attention.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4.0.
        dropout (float): Dropout rate. Default: 0.0.
        attention_dropout (float): Attention dropout rate. Default: 0.0.
        stochastic_depth_prob: (float): Stochastic depth rate. Default: 0.0.
        norm_layer (nn.Module): Normalization layer.  Default: nn.LayerNorm.
        attn_layer (nn.Module): Attention layer. Default: ShiftedWindowAttentionV2.
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        window_size: List[int],
        shift_size: List[int],
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        attention_dropout: float = 0.0,
        stochastic_depth_prob: float = 0.0,
        norm_layer: Callable[..., nn.Module] = nn.LayerNorm,
        attn_layer: Callable[..., nn.Module] = ShiftedWindowAttentionV2,
    ):
        super().__init__(
            dim,
            num_heads,
            window_size,
            shift_size,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
            attention_dropout=attention_dropout,
            stochastic_depth_prob=stochastic_depth_prob,
            norm_layer=norm_layer,
            attn_layer=attn_layer,
        )

    def forward(self, x: Tensor):
        # Here is the difference, we apply norm after the attention in V2.
        # In V1 we applied norm before the attention.
        x = x + self.stochastic_depth(self.norm1(self.attn(x)))
        x = x + self.stochastic_depth(self.norm2(self.mlp(x)))
        return x

In [81]:
# We need land mask to not attend over land! But how to construct it as we go deeper? Or maybe we just encode land very differently. Provide an option!

## Swin Transformer Encoder + Samudra Decoder 

## Samudra-Attention (in encoder only)
Some efficient swin transformers use convolutions for patch merging and patch embedding. Could be useful